# BT4012 Competition — Tuning on the 85-feature set

Checks the final model's settings on the current **85 features**, scored by **ROC-AUC**, the competition metric.

**Validation split: gap-matched.** Train on `time_step` 1–20, validate on 30–35. The final model trains through time step 35 and predicts 36–49, between 1 and 14 steps ahead. The adjacent 31–35 window sits right after training and scores ROC-AUC ≈ 0.998 for nearly every setting, so it can't separate them. Leaving a gap of 10–15 steps brings validation closer to the real distance and gives ROC-AUC room to separate settings. (This split proved reliable for training choices such as the half-life, but it favoured more flexible models that then lost on the leaderboard; see the outcome notes below.)

**Steps:**
1. Half-life sweep (none, 5, 10, 20, 40) with the current hyperparameters.
2. The 72-configuration XGBoost grid at the winning half-life.
3. Compare the best result with the current setup (`half_life=10` + current hyperparameters).
4. SHAP-based feature reduction (this produced the final 45-feature model).
5. A matching 72-configuration LightGBM grid.

Step 4 writes experiment files to `data/submissions/experiments/`; `data/submissions/final_submission.csv` is not touched.


In [1]:
import itertools

from lightgbm import LGBMClassifier
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier


## Data and gap-matched split

In [2]:
train_reduced = pd.read_csv("data/processed/train_reduced.csv")

NON_FEATURES = {"txId", "time_step", "label", "label_propagated", "label_propagated_confidence"}
feat_cols = [c for c in train_reduced.columns if c not in NON_FEATURES]

labeled = train_reduced[train_reduced["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)

GAP_TRAIN_CUTOFF = 20
train_gap = labeled[labeled["time_step"] <= GAP_TRAIN_CUTOFF]
val_gap = labeled[labeled["time_step"].between(30, 35)]

print("features:", len(feat_cols))
print(f"train (time_step 1-{GAP_TRAIN_CUTOFF}): {len(train_gap)} rows, illicit {train_gap['label'].mean():.2%}")
print(f"validation (time_step 30-35): {len(val_gap)} rows, illicit {val_gap['label'].mean():.2%}")

features: 85
train (time_step 1-20): 18889 rows, illicit 9.38%
validation (time_step 30-35): 4854 rows, illicit 15.93%


In [3]:
CURRENT_PARAMS = dict(max_depth=4, learning_rate=0.1, subsample=1.0, colsample_bytree=0.7, reg_lambda=5)
CURRENT_HALF_LIFE = 10


def sample_weight(rows, half_life):
    y = rows["label"]
    class_ratio = (y == 0).sum() / (y == 1).sum()
    weight = y.map({0: 1.0, 1: class_ratio})
    if half_life is not None:
        weight = weight * 0.5 ** ((rows["time_step"].max() - rows["time_step"]) / half_life)
    return weight


def score(params, half_life):
    model = XGBClassifier(n_estimators=300, eval_metric="logloss", random_state=42, n_jobs=-1, **params)
    model.fit(train_gap[feat_cols], train_gap["label"], sample_weight=sample_weight(train_gap, half_life))
    return roc_auc_score(val_gap["label"], model.predict_proba(val_gap[feat_cols])[:, 1])

## Step 1 — Half-life sweep

All runs use the current hyperparameters, so only the recency weighting changes.

In [4]:
half_life_results = pd.Series(
    {str(h): score(CURRENT_PARAMS, h) for h in [None, 5, 10, 20, 40]}, name="ROC-AUC"
).rename_axis("half_life")
best_half_life = int(half_life_results.drop("None").idxmax())

print(half_life_results.round(4).to_string())
print(f"\nbest half-life: {best_half_life}")

half_life
None    0.8538
5       0.8513
10      0.8809
20      0.8470
40      0.8342

best half-life: 10


## Step 2 — 72-configuration grid

Every combination of `max_depth` {4, 6, 8}, `learning_rate` {0.05, 0.1, 0.2}, `subsample` {0.7, 1.0}, `colsample_bytree` {0.7, 1.0} and `reg_lambda` {1, 5}, each with 300 trees, at the half-life chosen in step 1.

In [5]:
grid = {
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample": [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0],
    "reg_lambda": [1, 5],
}
rows = []
for combo in itertools.product(*grid.values()):
    params = dict(zip(grid.keys(), combo))
    rows.append({**params, "roc_auc": score(params, best_half_life)})

grid_results = pd.DataFrame(rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
grid_results.index = grid_results.index + 1
grid_results.index.name = "rank"
print(f"{len(grid_results)} configurations at half-life {best_half_life}")
grid_results.head(10).round(4)

72 configurations at half-life 10


,max_depth,learning_rate,subsample,colsample_bytree,reg_lambda,roc_auc
rank,,,,,,
1,4,0.20,1.0,1.0,1,0.9001
2,8,0.05,1.0,1.0,1,0.8987
3,6,0.20,0.7,1.0,5,0.8962
4,6,0.05,0.7,1.0,5,0.8950
5,4,0.20,0.7,0.7,1,0.8948
6,4,0.05,1.0,0.7,5,0.8937
7,4,0.05,1.0,1.0,5,0.8919
8,4,0.05,1.0,0.7,1,0.8899
9,8,0.05,0.7,1.0,5,0.8883


## Step 3 — Best result vs. the current setup

In [6]:
is_current = (grid_results[list(CURRENT_PARAMS)] == pd.Series(CURRENT_PARAMS)).all(axis=1)
current_score = score(CURRENT_PARAMS, CURRENT_HALF_LIFE)
best = grid_results.iloc[0]

print(f"current setup (half-life {CURRENT_HALF_LIFE}, current hyperparameters): ROC-AUC {current_score:.4f}")
print(f"current hyperparameters at half-life {best_half_life}: rank {grid_results.index[is_current.to_numpy()][0]} of {len(grid_results)}")
print(f"best found (half-life {best_half_life}, {dict(best[list(grid)])}): ROC-AUC {best['roc_auc']:.4f}")
print(f"difference: {best['roc_auc'] - current_score:+.4f}")

current setup (half-life 10, current hyperparameters): ROC-AUC 0.8809
current hyperparameters at half-life 10: rank 18 of 72
best found (half-life 10, {'max_depth': 4.0, 'learning_rate': 0.2, 'subsample': 1.0, 'colsample_bytree': 1.0, 'reg_lambda': 1.0}): ROC-AUC 0.9001
difference: +0.0192


## Outcome on the leaderboard

The half-life sweep confirmed `half_life=10` (also the winner on the 76-feature set), so that stays.

The top grid configuration (`max_depth=4, learning_rate=0.2, subsample=1.0, colsample_bytree=1.0, reg_lambda=1`) scored **0.94936** on the leaderboard, below the current settings' **0.95430**, despite its +0.019 gap-matched gain. The current, more regularized settings were kept. Picking the best of 72 noisy validation scores inflates the winner, and settings that generalize to `time_step` 30–35 don't necessarily hold for 36–49.

## Step 4 — SHAP-based feature reduction

Does dropping the features the model uses least help on the 85-feature set? An earlier test on 76 features found every cut worse than keeping all of them, but the feature set has changed since.

The ranking used for validation comes from a model trained only on the gap-matched training rows (`time_step` 1–20), with the current settings and `half_life=10`. A ranking from the final model would have seen the validation labels and leak them into which features are kept.

Cuts tested: keep the top 75 / 65 / 55 / 45 / 35 features, and a **light cut** that drops only features under 0.2% of total SHAP impact.

In [7]:
SHARE_FLOOR = 0.002


def fit(rows, cols):
    model = XGBClassifier(n_estimators=300, eval_metric="logloss", random_state=42, n_jobs=-1, **CURRENT_PARAMS)
    model.fit(rows[cols], rows["label"], sample_weight=sample_weight(rows, CURRENT_HALF_LIFE))
    return model


def shap_share(model, rows, cols):
    values = pd.Series(np.abs(shap.TreeExplainer(model).shap_values(rows[cols])).mean(axis=0), index=cols)
    return (values / values.sum()).sort_values(ascending=False)


def gap_score(cols):
    model = fit(train_gap, cols)
    return roc_auc_score(val_gap["label"], model.predict_proba(val_gap[cols])[:, 1])


gap_share = shap_share(fit(train_gap, feat_cols), train_gap, feat_cols)

reduction = {"all 85": (len(feat_cols), gap_score(feat_cols))}
for n in [75, 65, 55, 45, 35]:
    reduction[f"top {n}"] = (n, gap_score(list(gap_share.index[:n])))
light_cols = list(gap_share[gap_share >= SHARE_FLOOR].index)
reduction["light cut"] = (len(light_cols), gap_score(light_cols))

reduction_results = pd.DataFrame(reduction, index=["features", "ROC-AUC"]).T
reduction_results["features"] = reduction_results["features"].astype(int)
reduction_results.round(4)

,features,ROC-AUC
all 85,85,0.8809
top 75,75,0.8508
top 65,65,0.8591
top 55,55,0.8617
top 45,45,0.8819
top 35,35,0.8700
light cut,61,0.8700


### Submission files

Two cuts are exported for the leaderboard: the light cut, and the top-N cut that scored best above. For the final models the ranking comes from the model trained on all labeled data, since no validation is involved. The 85-feature model must reproduce `data/submissions/experiments/submission_85feat.csv` (leaderboard 0.95430) exactly, confirming the recipe matches the best submission. `data/submissions/final_submission.csv` is not touched.

In [8]:
test_reduced = pd.read_csv("data/processed/test_reduced.csv")

full_model = fit(labeled, feat_cols)
reference = pd.read_csv("data/submissions/experiments/submission_85feat.csv")["target"].to_numpy()
print("85-feature model reproduces submission_85feat.csv:",
      np.allclose(full_model.predict_proba(test_reduced[feat_cols])[:, 1], reference, atol=1e-6))

full_share = shap_share(full_model, labeled, feat_cols)
best_n = int(reduction_results.loc[[k for k in reduction_results.index if k.startswith("top ")], "ROC-AUC"]
             .idxmax().split()[1])

exports = {
    "submission_shap_light.csv": list(full_share[full_share >= SHARE_FLOOR].index),
    f"submission_shap_top{best_n}.csv": list(full_share.index[:best_n]),
}
for filename, cols in exports.items():
    model = fit(labeled, cols)
    target = model.predict_proba(test_reduced[cols])[:, 1]
    pd.DataFrame({"index": np.arange(len(test_reduced)), "target": target}).to_csv(f"data/submissions/experiments/{filename}", index=False)
    print(f"saved data/submissions/experiments/{filename}: {len(cols)} features")

dropped = sorted(set(feat_cols) - set(exports["submission_shap_light.csv"]))
print(f"\nlight cut drops {len(dropped)} features:", dropped)

85-feature model reproduces submission_85feat.csv: True
saved data/submissions/experiments/submission_shap_light.csv: 58 features
saved data/submissions/experiments/submission_shap_top45.csv: 45 features

light cut drops 27 features: ['degree_diff', 'feat_105', 'feat_110', 'feat_117', 'feat_120', 'feat_123', 'feat_129', 'feat_141', 'feat_147', 'feat_15', 'feat_150', 'feat_153', 'feat_160', 'feat_32', 'feat_35', 'feat_36', 'feat_38', 'feat_4', 'feat_45', 'feat_63', 'feat_7', 'feat_70', 'feat_75', 'feat_82', 'feat_98', 'flow_ratio', 'total_degree']


## Step 5 — LightGBM hyperparameter tuning

A 72-configuration grid mirroring the XGBoost one, on all 85 features, with the same `half_life=10` recency and class weights, on the gap-matched split. `num_leaves` replaces `max_depth` as LightGBM's main complexity control (7, 15 and 31 leaves roughly match depths 3, 4 and 5). `subsample` re-samples rows on every tree.

| Setting | Values |
|---|---|
| `num_leaves` | 7, 15, 31 |
| `learning_rate` | 0.05, 0.1, 0.2 |
| `subsample` | 0.7, 1.0 |
| `colsample_bytree` | 0.7, 1.0 |
| `reg_lambda` | 1, 5 |

Each configuration uses 300 trees and is compared with the final XGBoost recipe on the same split (step 3).

In [9]:
lgbm_grid = {
    "num_leaves": [7, 15, 31],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample": [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0],
    "reg_lambda": [1, 5],
}
gap_weight = sample_weight(train_gap, CURRENT_HALF_LIFE)

lgbm_rows = []
for combo in itertools.product(*lgbm_grid.values()):
    params = dict(zip(lgbm_grid, combo))
    model = LGBMClassifier(n_estimators=300, subsample_freq=1, random_state=42, n_jobs=-1, verbose=-1, **params)
    model.fit(train_gap[feat_cols], train_gap["label"], sample_weight=gap_weight)
    lgbm_rows.append({**params, "roc_auc": roc_auc_score(val_gap["label"], model.predict_proba(val_gap[feat_cols])[:, 1])})

lgbm_results = pd.DataFrame(lgbm_rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
lgbm_results.index = lgbm_results.index + 1
lgbm_results.index.name = "rank"

best_lgbm = lgbm_results["roc_auc"].iloc[0]
print(f"XGBoost final recipe on this split: ROC-AUC {current_score:.4f}")
print(f"best LightGBM configuration:        ROC-AUC {best_lgbm:.4f} ({best_lgbm - current_score:+.4f})")
print(f"LightGBM configurations beating XGBoost: {(lgbm_results['roc_auc'] > current_score).sum()} of {len(lgbm_results)}")
lgbm_results.head(10).round(4)

XGBoost final recipe on this split: ROC-AUC 0.8809
best LightGBM configuration:        ROC-AUC 0.8964 (+0.0155)
LightGBM configurations beating XGBoost: 5 of 72


,num_leaves,learning_rate,subsample,colsample_bytree,reg_lambda,roc_auc
rank,,,,,,
1,31,0.20,0.7,0.7,1,0.8964
2,7,0.05,1.0,0.7,5,0.8902
3,15,0.05,1.0,0.7,1,0.8886
4,7,0.05,1.0,0.7,1,0.8869
5,7,0.05,1.0,1.0,1,0.8818
6,7,0.20,1.0,0.7,1,0.8804
7,7,0.05,0.7,1.0,1,0.8792
8,7,0.10,1.0,0.7,1,0.8774
9,7,0.10,1.0,1.0,5,0.8773
